In [10]:
from pathlib import Path
from dataclasses import dataclass, replace
import dataclasses
import json
import re

import kdrag.solvers.tla as tla
from hypothesis import given, settings, strategies as st

Connect python functions to TLA first? Hypothesis testing?

It could be kind of fun to mock locking or something to emit action labels / trace events. This could be somewhat transparent (opaque? depends on your preferred term)


In [4]:
%%file /tmp/HourClock.tla
---- MODULE HourClock ----
EXTENDS Naturals

VARIABLE hr

HCini == hr \in 1 .. 12
HCnxt == hr' = IF hr = 12 THEN 1 ELSE hr + 1
Next == HCnxt \/ UNCHANGED hr
HC == HCini /\ [][HCnxt]_hr
====

Overwriting /tmp/HourClock.tla


In [5]:
%%file /tmp/HourClock.cfg
INIT HCini
NEXT Next

Overwriting /tmp/HourClock.cfg


In [7]:
@dataclass(frozen=True)
class ClockState:
    hr: int


def tick(state):
    return replace(state, hr=state.hr % 13 + 1)


def stutter(state):
    return replace(state)


actions = {"tick": tick, "stutter": stutter}


@st.composite
def hourclock_traces(draw, max_steps=10):
    state = ClockState(draw(st.integers(1, 12)))
    trace = [state]
    names = draw(st.lists(st.sampled_from(list(actions)), max_size=max_steps))
    for name in names:
        state = actions[name](state)
        trace.append(state)
    return trace

In [13]:
%%prun
def validate_trace(trace : list[ClockState]) -> bool:
    states = [dataclasses.asdict(state) for state in trace]
    data = {
        "vars": ["hr"],
        "counterexample": {
            "state": [[i, state] for i, state in enumerate(states, 1)],
            "action": [],
        },
    }
    tracefile = "/tmp/hourclock_trace.json"
    Path(tracefile).write_text(json.dumps(data))
    out = tla.run_tools([
        "tlc2.TLC",
        "-workers", "1",
        "-loadTrace", "json", tracefile,
        "-config", "/tmp/HourClock",
        "/tmp/HourClock.tla",
    ]).decode()
    print(out)
    #assert "No error has been found." in out, "Trace is invalid"
    depth = int(re.search(r"The depth .* is (\d+)\.", out).group(1))
    return depth >= len(trace)


assert validate_trace([ClockState(12), ClockState(1), ClockState(1)])
assert not validate_trace([ClockState(1), ClockState(3)])

TLC2 Version 2026.07.14.071606 (rev: 227f61b)
(Use the -nowarning option to disable this warning.)
Running breadth-first search Model-Checking with fp 103 and seed 1047999301453507687 with 1 worker on 16 cores with 15207MB heap and 64MB offheap memory [pid: 1334020] (Linux 7.0.0-28-generic amd64, Ubuntu 21.0.11 64bit, MSBDiskFPSet, DiskStateQueue).
Parsing file /tmp/HourClock.tla
Parsing file /tmp/tlc-8557857232466255485/Naturals.tla (jar:file:/home/philip/vibe_coding/knuck_anal/knuckledragger/src/kdrag/solvers/tla2tools.jar!/tla2sany/StandardModules/Naturals.tla)
Parsing file /tmp/tlc-8557857232466255485/_TLCTrace.tla (jar:file:/home/philip/vibe_coding/knuck_anal/knuckledragger/src/kdrag/solvers/tla2tools.jar!/tla2sany/StandardModules/_TLCTrace.tla)
Parsing file /tmp/tlc-8557857232466255485/_JsonTrace.tla (jar:file:/home/philip/vibe_coding/knuck_anal/knuckledragger/src/kdrag/solvers/tla2tools.jar!/tla2sany/StandardModules/_JsonTrace.tla)
Parsing file /tmp/tlc-8557857232466255485/TLC.t

         3569 function calls (3551 primitive calls) in 1.515 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
       78    1.469    0.019    1.469    0.019 {method 'poll' of 'select.poll' objects}
        4    0.034    0.008    0.034    0.008 {method 'poll' of 'select.epoll' objects}
        2    0.002    0.001    0.002    0.001 {method '__exit__' of 'sqlite3.Connection' objects}
       78    0.002    0.000    1.471    0.019 selectors.py:402(select)
        2    0.001    0.001    1.309    0.654 subprocess.py:2062(_communicate)
        2    0.001    0.000    0.001    0.000 {built-in method _posixsubprocess.fork_exec}
       82    0.001    0.000    0.001    0.000 {built-in method posix.read}
        2    0.001    0.000    0.001    0.000 {built-in method posix.waitpid}
       23    0.000    0.000    0.000    0.000 socket.py:623(send)
        2    0.000    0.000    0.002    0.001 subprocess.py:807(__init__)
       82    0.000   

In [9]:
@settings(max_examples=10, deadline=None)
@given(hourclock_traces())
def test_python_hourclock_refines_tla(trace):
    assert validate_trace(trace), trace


test_python_hourclock_refines_tla()

AssertionError: [ClockState(hr=9), ClockState(hr=10), ClockState(hr=11), ClockState(hr=12), ClockState(hr=13)]

cerberus
cbmc

renode
actually control the hardware? gdb scrpit from python

https://arxiv.org/abs/2404.16075 merz Validating Traces of Distributed Programs Against TLA+ Specifications

https://www.youtube.com/watch?v=NZmON-XmrkI Validating System Executions with the TLA+ Tools Markus A Kuppe, Microsoft

https://www.youtube.com/watch?v=W6DrQk8o5tk

https://docs.tlapl.us/using:tlc:trace_validation 

https://pron.github.io/files/Trace.pdf ron pressler trace vliation 2018

It's surprising there is an json tlc module. also an IO module?

tla importer could wrap exprssion in dummy module.
```python
def expr(e : str, variables=[], constants=[]):
    with write() as f:
        f.write("----- KDRAGDUMMY --------)
        f.write(f"VARIABLES {v})
        f.write(f"KDRAGEXPR == {e}\n")
        f.write(f"==================")
    mod = Module.load_file("/tmp/KDRAGDUMMY.tla")
    mod.infer_sorts()
    return mod.action("KDRAGEXPR")
```

Yea, maybe I'm getting closer to SPIN?

cocotb might be kind of interesting...
spike or sail derived emulator?
Try a bunch of them?




In [39]:
%%file /tmp/hour.c

#include <stdio.h>
#include <stdlib.h>
#include <time.h>   

typedef struct ClockState {
    int hr;
} ClockState;

ClockState state;

void tick(){
    state.hr = state.hr % 13 + 1;
}

void main(){
    srand(time(NULL));
    state.hr = rand() % 12 + 1;
    printf("[");
    for(int t = 0; t < 100; t++){
        printf("[%d, { hr : %d }]\n", t, state.hr);
        tick();
    }
    printf("]");
}


Overwriting /tmp/hour.c


In [40]:
! gcc -o /tmp/hour /tmp/hour.c && /tmp/hour

[[0, { hr : 3 }]
[1, { hr : 4 }]
[2, { hr : 5 }]
[3, { hr : 6 }]
[4, { hr : 7 }]
[5, { hr : 8 }]
[6, { hr : 9 }]
[7, { hr : 10 }]
[8, { hr : 11 }]
[9, { hr : 12 }]
[10, { hr : 13 }]
[11, { hr : 1 }]
[12, { hr : 2 }]
[13, { hr : 3 }]
[14, { hr : 4 }]
[15, { hr : 5 }]
[16, { hr : 6 }]
[17, { hr : 7 }]
[18, { hr : 8 }]
[19, { hr : 9 }]
[20, { hr : 10 }]
[21, { hr : 11 }]
[22, { hr : 12 }]
[23, { hr : 13 }]
[24, { hr : 1 }]
[25, { hr : 2 }]
[26, { hr : 3 }]
[27, { hr : 4 }]
[28, { hr : 5 }]
[29, { hr : 6 }]
[30, { hr : 7 }]
[31, { hr : 8 }]
[32, { hr : 9 }]
[33, { hr : 10 }]
[34, { hr : 11 }]
[35, { hr : 12 }]
[36, { hr : 13 }]
[37, { hr : 1 }]
[38, { hr : 2 }]
[39, { hr : 3 }]
[40, { hr : 4 }]
[41, { hr : 5 }]
[42, { hr : 6 }]
[43, { hr : 7 }]
[44, { hr : 8 }]
[45, { hr : 9 }]
[46, { hr : 10 }]
[47, { hr : 11 }]
[48, { hr : 12 }]
[49, { hr : 13 }]
[50, { hr : 1 }]
[51, { hr : 2 }]
[52, { hr : 3 }]
[53, { hr : 4 }]
[54, { hr : 5 }]
[55, { hr : 6 }]
[56, { hr : 7 }]
[57, { hr : 8 }]
[58, { 

In [48]:
%%file /tmp/hour.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
  
typedef struct ClockState {
    int hr;
} ClockState;

ClockState state;

void tick(){
    state.hr = state.hr % 13 + 1;
}

int main(){
    srand(time(NULL));
    state.hr = rand() % 12 + 1;
    for(int t = 0; t < 100; t++){
        tick();
    }
    return 0;
}

Overwriting /tmp/hour.c


In [49]:
! gcc -g -Wall -o /tmp/hour /tmp/hour.c

import gdb only works inside GDB's embedded Python.
 GDB/MI i  https://sourceware.org/gdb/current/onlinedocs/gdb.html/GDB_002fMI.html
 Is this overwrought?
 Should I just make a python scriper and load it from inside gdb or script gdb in some other way


 https://www.youtube.com/watch?v=xt9v5t4_zvE lisa roach - extended gdb with python. Very fun
Could this be a road into some whackasmackadoo tower of interpreters stuff?


Hmm. Control renode via gdb? https://renode.readthedocs.io/en/latest/debugging/gdb.html
https://github.com/matgla/Renode_RP2040
https://github.com/wokwi/rp2040js


Worry: instrumentation may change system. printf has locks in pico for example
Make instrumentation so cheap you leave it on? (antithesis right?)
Or further testing required anyhow


In [ ]:
import os
os.getpid()

In [ ]:
%%file /tmp/printhello.py

import gdb
gdb.execute("call \"Python)



In [ ]:
%%file /tmp/hourclock.gdb
set pagination off
set confirm off
set debuginfod enabled off
break tick
commands
  silent
  printf "CLOCK %d\n", state.hr
  continue
end
run

In [ ]:
import subprocess

result = subprocess.run(
    ["gdb", "-q", "--batch", "-x", "/tmp/hourclock.gdb", "/tmp/hour"],
    capture_output=True, text=True, check=True,
)
c_trace = [
    ClockState(int(hr))
    for hr in re.findall(r"^CLOCK (\d+)$", result.stdout, re.MULTILINE)
]
c_trace[:10], len(c_trace)

In [58]:
import sys
sys.version
sys.executable


'/home/philip/philzook58.github.io/.venv/bin/python'

1285037

In [56]:
! gdb -ex "python import sys; print(sys.version); print(sys.executable)" -ex "quit"

GNU gdb (Ubuntu 15.1-1ubuntu1~24.04.1) 15.1
Copyright (C) 2024 Free Software Foundation, Inc.
License GPLv3+: GNU GPL version 3 or later <http://gnu.org/licenses/gpl.html>
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.
Type "show copying" and "show warranty" for details.
This GDB was configured as "x86_64-linux-gnu".
Type "show configuration" for configuration details.
For bug reporting instructions, please see:
<https://www.gnu.org/software/gdb/bugs/>.
Find the GDB manual and other documentation resources online at:
    <http://www.gnu.org/software/gdb/documentation/>.

For help, type "help".
Type "apropos word" to search for commands related to "word".
3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]
/usr/bin/python


# Renode


In [61]:
%%file /tmp/hourclock_rv.c
typedef struct { volatile unsigned int hr; } ClockState;
volatile ClockState state = {1};
extern char __stack_top[];
int main(void);

__attribute__((naked, section(".text.start")))
void _start(void) {
    __asm__ volatile("la sp, __stack_top\n"
                     "call main\n"
                     "ebreak\n"
                     "1: j 1b");
}

__attribute__((noinline)) void tick(void) { state.hr = state.hr % 12 + 1; }
__attribute__((noinline)) void tick_done(void) {}

int main(void) {
    for(int i = 0; i < 10; i++) {
        tick();
        tick_done();
    }
    return 0;
}


Overwriting /tmp/hourclock_rv.c


In [62]:
%%file /tmp/hourclock_rv.ld
ENTRY(_start)
SECTIONS {
    . = 0x80000000;
    .text : { KEEP(*(.text.start)) *(.text*) }
    .rodata : { *(.rodata*) }
    .data : { *(.data*) }
    .bss : { *(.bss*) *(COMMON) }
    . = ALIGN(16);
    . += 0x1000;
    __stack_top = .;
}


Overwriting /tmp/hourclock_rv.ld


In [2]:
%%file /tmp/hourclock.resc
mach create "hourclock"
machine LoadPlatformDescriptionFromString """
cpu: CPU.RiscV64 @ sysbus
    cpuType: "rv64imac"
    privilegedArchitecture: PrivilegedArchitecture.Priv1_12
    timeProvider: empty

ram: Memory.MappedMemory @ sysbus 0x80000000
    size: 0x100000
"""
sysbus LoadELF @/tmp/hourclock_rv.elf
machine StartGdbServer 3333


Overwriting /tmp/hourclock.resc


In [64]:
%%file /tmp/hourclock_renode.py
import gdb
import json

trace = []

class TickDone(gdb.Breakpoint):
    def stop(self):
        trace.append({"hr": int(gdb.parse_and_eval("state.hr"))})
        return False

TickDone("tick_done")
gdb.execute("monitor start")
gdb.execute("continue")
print("CLOCKTRACE " + json.dumps(trace))


Overwriting /tmp/hourclock_renode.py


In [65]:
!riscv64-unknown-elf-gcc -march=rv64imac -mabi=lp64 -mcmodel=medany \
    -g -O0 -ffreestanding -nostdlib -Wl,-T,/tmp/hourclock_rv.ld \
    -o /tmp/hourclock_rv.elf /tmp/hourclock_rv.c


/usr/lib/gcc/riscv64-unknown-elf/13.2.0/../../../riscv64-unknown-elf/bin/ld: warning: /tmp/hourclock_rv.elf has a LOAD segment with RWX permissions


In [3]:
import json
import re
import subprocess

renode = subprocess.Popen(
    ["dotnet", "/opt/renode/bin/Renode.dll",
     "--disable-xwt", "--plain", "--config",
     "/tmp/hourclock-renode-config", "/tmp/hourclock.resc"],
    stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True,
)
try:
    result = subprocess.run([
        "gdb-multiarch", "-q", "--batch", "/tmp/hourclock_rv.elf",
        "-ex", "target remote :3333",
        "-ex", "source /tmp/hourclock_renode.py",
    ], capture_output=True, text=True, timeout=20)
    result.check_returncode()
    match = re.search(r"^CLOCKTRACE (.*)$", result.stdout, re.MULTILINE)
    assert match, result.stdout + result.stderr
    renode_trace = [ClockState(**st) for st in json.loads(match.group(1))]
finally:
    renode.terminate()
    renode.wait(timeout=5)

renode_trace


TimeoutExpired: Command '['gdb-multiarch', '-q', '--batch', '/tmp/hourclock_rv.elf', '-ex', 'target remote :3333', '-ex', 'source /tmp/hourclock_renode.py']' timed out after 20 seconds

# Pico trace

interrupt triggering?
send over hypothesis generated interrupt schedule?
hardware watchpoints

Could ingest TLA spec and directly look for violations in the fuzzer.
python script could generate hypotheses itself and just stream out traces.

If we ingest TLA spec, instead of using TLC, could check directly in python. Best to do both? Maybe being in python could inform hypothesis more (in interview they mentioned it peeks at source code?)

```
import tla
tla.Module.of_()


```

Errors in gdb script

TICKSTART TICKEND. Maybe actions are kind of spread over time not an instant?












In [ ]:
Path("/tmp/picohour").mkdir(exist_ok=True)

In [7]:
%%file /tmp/picohour/CMakeLists.txt
cmake_minimum_required(VERSION 3.13)
set(PICO_BOARD pico2)
set(PICO_SDK_PATH /home/philip/.pico-sdk/sdk/2.3.0)
set(PICO_TOOLCHAIN_PATH /home/philip/.pico-sdk/toolchain/15_2_Rel1)
set(picotool_DIR /home/philip/.pico-sdk/picotool/2.3.0/picotool)
include(/home/philip/.pico-sdk/sdk/2.3.0/external/pico_sdk_import.cmake)

project(hourclock C CXX ASM)
pico_sdk_init()

add_executable(hourclock hourclock.c)
target_link_libraries(hourclock pico_stdlib)

Overwriting /tmp/picohour/CMakeLists.txt


In [8]:
%%file /tmp/picohour/hourclock.c
#include "pico/stdlib.h"

typedef struct { volatile unsigned int hr; } ClockState;
volatile ClockState state = {1};

__attribute__((noinline)) void trace_point(void) { __asm volatile ("nop"); }
__attribute__((noinline)) void trace_done(void) { __asm volatile ("nop"); }

int main(void) {
    trace_point();
    for (int i = 0; i < 10; i++) {
        state.hr = state.hr % 12 + 1;
        trace_point();
    }
    trace_done();
    while (true) tight_loop_contents();
}


Overwriting /tmp/picohour/hourclock.c


In [11]:
import subprocess
subprocess.run(["cmake", "-S", "/tmp/picohour", "-B", "/tmp/picohour/build",
                "-G", "Ninja", "-DCMAKE_BUILD_TYPE=Debug"], check=True)
subprocess.run(["cmake", "--build", "/tmp/picohour/build"], check=True)

PICO_SDK_PATH is /home/philip/.pico-sdk/sdk/2.3.0
Target board (PICO_BOARD) is 'pico2'.
Using board configuration from /home/philip/.pico-sdk/sdk/2.3.0/src/boards/include/boards/pico2.h
Pico Platform (PICO_PLATFORM) is 'rp2350-arm-s'.


-- The C compiler identification is GNU 13.2.1
-- The CXX compiler identification is GNU 13.2.1
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/arm-none-eabi-gcc
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/arm-none-eabi-gcc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/arm-none-eabi-g++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done


Build type is Debug
Using regular optimized debug build (set PICO_DEOPTIMIZED_DEBUG=1 to de-optimize)
Using picotool from /home/philip/.pico-sdk/picotool/2.3.0/picotool/picotool


-- Found Python3: /home/philip/philzook58.github.io/.venv/bin/python3.12 (found version "3.12.3") found components: Interpreter
-- Configuring done (0.7s)


TinyUSB available at /home/philip/.pico-sdk/sdk/2.3.0/lib/tinyusb/hw/bsp/rp2040; enabling build support for USB.
Compiling TinyUSB with CFG_TUSB_DEBUG=1
BTstack available at /home/philip/.pico-sdk/sdk/2.3.0/lib/btstack
cyw43-driver available at /home/philip/.pico-sdk/sdk/2.3.0/lib/cyw43-driver
mbedtls available at /home/philip/.pico-sdk/sdk/2.3.0/lib/mbedtls
lwIP available at /home/philip/.pico-sdk/sdk/2.3.0/lib/lwip
C library type is newlib


-- Generating done (0.1s)
-- Build files have been written to: /tmp/picohour/build
[1/4] Generating bs2_default_padded.S
[2/4] Building ASM object pico-sdk/src/rp2350/boot_stage2/CMakeFiles/bs2_default_library.dir/bs2_default_padded.S.o
[3/4] Building C object CMakeFiles/hourclock.dir/hourclock.c.o
[4/4] Linking CXX executable hourclock.elf


CompletedProcess(args=['cmake', '--build', '/tmp/picohour/build'], returncode=0)

In [12]:
%%file /tmp/picohour/trace.py

import gdb
import json

trace = []

class TracePoint(gdb.Breakpoint):
    def stop(self):
        trace.append({"hr": int(gdb.parse_and_eval("state.hr"))})
        return False

TracePoint("trace_point")
gdb.Breakpoint("trace_done")
gdb.execute("monitor reset init")
gdb.execute("load")
gdb.execute("continue")
print("CLOCKTRACE " + json.dumps(trace))


Overwriting /tmp/picohour/trace.py


In [16]:
import re
import json
from dataclasses import dataclass, replace
@dataclass
class ClockState:
    hr : int
openocd = subprocess.Popen([
    "/home/philip/.pico-sdk/openocd/0.12.0+dev/openocd",
    "-s", "/home/philip/.pico-sdk/openocd/0.12.0+dev/scripts",
    "-f", "interface/cmsis-dap.cfg", "-f", "target/rp2350.cfg",
    "-c", "adapter speed 5000",
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in openocd.stdout:
        if "Listening on port 3333 for gdb connections" in line:
            break
    result = subprocess.run([
        "gdb-multiarch", "-q", "--batch", "/tmp/picohour/build/hourclock.elf",
        "-ex", "target extended-remote localhost:3333",
        "-ex", "source /tmp/picohour/trace.py",
    ], capture_output=True, text=True, check=True, timeout=30)
finally:
    openocd.terminate()
    openocd.wait(timeout=5)

match = re.search(r"^CLOCKTRACE (.*)$", result.stdout, re.MULTILINE)
assert match, result.stdout + result.stderr
pico_trace = [ClockState(**st) for st in json.loads(match.group(1))]
pico_trace

[ClockState(hr=1),
 ClockState(hr=2),
 ClockState(hr=3),
 ClockState(hr=4),
 ClockState(hr=5),
 ClockState(hr=6),
 ClockState(hr=7),
 ClockState(hr=8),
 ClockState(hr=9),
 ClockState(hr=10),
 ClockState(hr=11)]

# rust



In [ ]:
%%file /tmp/hourclock.rs

struct ClockState {
    hr: u32,
}
impl ClockState {
    fn new(hr: u32) -> Self {
        assert!(hr >= 1 && hr <= 12);
        ClockState { hr }
    }
    fn tick(&mut self) {
        self.hr = (self.hr + 1) % 12;
    }
}

fn main(){
    let mut clock = ClockState::new(12);
    for i in 0..100 {
        clock.tick();
        println!("CLOCK {} {{ hr : {} }}", i, clock.hr);
    }
}


Overwriting /tmp/hourclock.rs


In [23]:
! rustc -g -C debuginfo=2 -o /tmp/hourclock /tmp/hourclock.rs && /tmp/hourclock

CLOCK 0 { hr : 1 }
CLOCK 1 { hr : 2 }
CLOCK 2 { hr : 3 }
CLOCK 3 { hr : 4 }
CLOCK 4 { hr : 5 }
CLOCK 5 { hr : 6 }
CLOCK 6 { hr : 7 }
CLOCK 7 { hr : 8 }
CLOCK 8 { hr : 9 }
CLOCK 9 { hr : 10 }
CLOCK 10 { hr : 11 }
CLOCK 11 { hr : 0 }
CLOCK 12 { hr : 1 }
CLOCK 13 { hr : 2 }
CLOCK 14 { hr : 3 }
CLOCK 15 { hr : 4 }
CLOCK 16 { hr : 5 }
CLOCK 17 { hr : 6 }
CLOCK 18 { hr : 7 }
CLOCK 19 { hr : 8 }
CLOCK 20 { hr : 9 }
CLOCK 21 { hr : 10 }
CLOCK 22 { hr : 11 }
CLOCK 23 { hr : 0 }
CLOCK 24 { hr : 1 }
CLOCK 25 { hr : 2 }
CLOCK 26 { hr : 3 }
CLOCK 27 { hr : 4 }
CLOCK 28 { hr : 5 }
CLOCK 29 { hr : 6 }
CLOCK 30 { hr : 7 }
CLOCK 31 { hr : 8 }
CLOCK 32 { hr : 9 }
CLOCK 33 { hr : 10 }
CLOCK 34 { hr : 11 }
CLOCK 35 { hr : 0 }
CLOCK 36 { hr : 1 }
CLOCK 37 { hr : 2 }
CLOCK 38 { hr : 3 }
CLOCK 39 { hr : 4 }
CLOCK 40 { hr : 5 }
CLOCK 41 { hr : 6 }
CLOCK 42 { hr : 7 }
CLOCK 43 { hr : 8 }
CLOCK 44 { hr : 9 }
CLOCK 45 { hr : 10 }
CLOCK 46 { hr : 11 }
CLOCK 47 { hr : 0 }
CLOCK 48 { hr : 1 }
CLOCK 49 { hr : 2 }
CL

In [110]:
%%file /tmp/hourclock_gdb.py
import gdb
#print("hello world")


gdb.write("hello world\n")

gdb.execute("set pagination off")
gdb.execute("file /tmp/hourclock")
gdb.execute("info functions")
#gdb.execute("list Clockstate.tick")

# shell commands
res = gdb.execute("! ls")
print(res) # nothin. Ok
gdb.execute("set confirm off")
gdb.execute("set debuginfod enabled off")
#gdb.execute("start")
gdb.execute("run") # run > /tmp/myoutfile
gdb.execute("quit")


Overwriting /tmp/hourclock_gdb.py


In [111]:
! gdb -ex "source /tmp/hourclock_gdb.py"    # /tmp/hourclock -ex "quit"

GNU gdb (Ubuntu 15.1-1ubuntu1~24.04.1) 15.1
Copyright (C) 2024 Free Software Foundation, Inc.
License GPLv3+: GNU GPL version 3 or later <http://gnu.org/licenses/gpl.html>
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.
Type "show copying" and "show warranty" for details.
This GDB was configured as "x86_64-linux-gnu".
Type "show configuration" for configuration details.
For bug reporting instructions, please see:
<https://www.gnu.org/software/gdb/bugs/>.
Find the GDB manual and other documentation resources online at:
    <http://www.gnu.org/software/gdb/documentation/>.

For help, type "help".
Type "apropos word" to search for commands related to "word".
hello world
of file /tmp/hourclock.
Use `info auto-load python-scripts [REGEXP]' to list them.
All defined functions:

File /home/philip/.rustup/toolchains/stable-x86_64-unknown-linux-gnu/lib/rustlib/src/rust/library/core/src/fmt/mod.rs:
729:	static fn core::fmt:

# Qemu
